In [ ]:
%config InlineBackend.figure_format = 'svg'

In [ ]:
import os
import random
import time 
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap.umap_ as umap
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from node2vec import Node2Vec
from tqdm import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import CategoricalAccuracy
import torch
from torch_geometric.data import Data
import spektral
from spektral.layers import GCNConv, GATConv
from spektral.layers import GraphSageConv
from spektral.data import Graph, Dataset, BatchLoader
from scipy.sparse import csr_matrix
from spektral.datasets import Cora
from torch_geometric.nn import DeepGraphInfomax, VGAE
from torch_geometric.utils import from_networkx
import scipy.sparse as sp
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from scipy.sparse.csgraph import laplacian
from scipy.sparse.linalg import eigsh
from collections import Counter
from sklearn.preprocessing import normalize
from joblib import Parallel, delayed
from torch_geometric.nn import GCNConv as PyG_GCNConv, VGAE as PyG_VGAE
from torch_geometric.data import Data

In [ ]:
SEED = 42

# Set seed for Python's built-in random module
random.seed(SEED)

# Set seed for NumPy
np.random.seed(SEED)

# Set seed for TensorFlow
tf.random.set_seed(SEED)

# Set seed for PyTorch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Create a custom Dataset for the graph
class CoraDataset(Dataset):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def read(self):
        data = Cora()  # Load the dataset
        graph = data.graphs[0]  # Access the first graph in the dataset
        return [Graph(x=graph.x, a=graph.a, y=graph.y)]

In [ ]:
embedding_dimensionality=150

In [ ]:
def convert_to_networkx(A):
    return nx.from_scipy_sparse_array(A)

In [ ]:
dataset = CoraDataset()
ground_truth_labels = dataset[0].y
labels=np.argmax(ground_truth_labels,axis=1)

In [ ]:
mask_file = "C:/Users/user/Benchmarking_fuse/masks/Cora/70_30/Cora_70_30_masked_indices_seed42.npy"
labels_to_be_masked = np.load(mask_file)

In [ ]:
len(labels_to_be_masked)

In [ ]:
masked_labels=[]
for i in np.arange(len(labels)):
    if i in labels_to_be_masked:
        masked_labels.append(-1)
    else:
        masked_labels.append(labels[i])
masked_labels=np.array(masked_labels)

In [ ]:
label_mask = masked_labels != -1

In [ ]:
X = dataset[0].x
A = dataset[0].a
G = convert_to_networkx(A)

In [ ]:
print("Adjacency Matrix Shape:", A.shape)
print("Graph Nodes:", G.number_of_nodes())
print("Graph Edges:", G.number_of_edges())

In [ ]:
# Convert your preprocessed data into a PyTorch Geometric Data object
X_py = Data(
    x=torch.tensor(X, dtype=torch.float),  # Node features
    edge_index=torch.tensor(np.array(A.nonzero()), dtype=torch.long),  # Edge indices
    y=torch.tensor(labels, dtype=torch.long)  # Labels
)

# Ensure edge_index is in the correct shape (2, num_edges)
X_py.edge_index = X_py.edge_index.to(torch.long)

## Embeddings

In [ ]:
import os
import pickle
import numpy as np
import tensorflow as tf

# -------------------------------------
# Configuration
# -------------------------------------
SEED = 42
dataset_name = "cora"
split_folder = "70-30"

# Input embeddings directory
load_dir = f"C:/Users/user/Benchmarking_fuse/benchmark_outputs/{dataset_name}/{split_folder}/"

# Output save directory
save_dir = f"./cora_analysis_results/embeddings/{split_folder.replace('-', '_')}/"
os.makedirs(save_dir, exist_ok=True)

# Embedding filenames to load
embedding_files = {
    "deepwalk": f"deepwalk_embedding_70_30_{SEED}.pkl",
    "node2vec": f"node2vec_embedding_70_30_{SEED}.pkl",
    "fuse": f"fuse_embedding_70_30_{SEED}.pkl",
    "vgae": f"vgae_embedding_70_30_{SEED}.pkl",
    "dgi": f"dgi_embedding_70_30_{SEED}.pkl",
    "random": f"random_embedding_70_30_{SEED}.pkl",
    "given": f"given_embedding_70_30_{SEED}.pkl"
}

# Dictionary to store embeddings
embedding_dict = {}

# -------------------------------------
# Utility: convert numpy to tf.Tensor
# -------------------------------------
def to_tf_tensor(x):
    """Convert numpy array to TensorFlow tensor."""
    if isinstance(x, tf.Tensor):
        return x
    return tf.convert_to_tensor(x, dtype=tf.float32)

# -------------------------------------
# Load and re-save embeddings
# -------------------------------------
for name, filename in embedding_files.items():
    filepath = os.path.join(load_dir, filename)
    if not os.path.exists(filepath):
        print(f" Warning: {name} embedding file not found at {filepath}. Skipping.")
        continue

    print(f" Loading {name.upper()} embedding from {filepath}...")
    with open(filepath, "rb") as f:
        emb = pickle.load(f)

    # Convert to TensorFlow tensor
    embedding_dict[name] = to_tf_tensor(np.array(emb, dtype=float))

    # Save again to new organized directory
    save_path = os.path.join(save_dir, f"{name}_embedding_70_30_{SEED}.pkl")
    with open(save_path, "wb") as f:
        pickle.dump(embedding_dict[name].numpy(), f)
    print(f" Saved {name.upper()} embedding to {save_path}")

print("\n All embeddings loaded into memory and re-saved successfully.")


## Helper functions

In [ ]:
def visualize_all_embeddings(all_embeddings, labels, label_mask):
    """
    Visualize all embeddings in a grid with 4 columns per row using UMAP.

    Parameters:
    - all_embeddings: Dictionary where keys are embedding methods, and values are embeddings.
    - labels: Labels (numpy array of shape [n_nodes]).
    - label_mask: Boolean array indicating known labels (True for known, False for unknown).
    """
    num_embeddings = len(all_embeddings)
    num_rows = (num_embeddings + 3) // 4  # Ensure enough rows for all embeddings
    fig, axes = plt.subplots(num_rows, 4, figsize=(8.27, 11.69))  # A4 size

    for i, (embedding_type, embedding) in tqdm(enumerate(all_embeddings.items()), 
                                               total=num_embeddings, desc="Visualizing embeddings"):
        row, col = divmod(i, 4)
        ax = axes[row, col] if num_rows > 1 else axes[col]  # Adjust for single-row case

        # Ensure embedding is a NumPy array
        if isinstance(embedding, tf.Tensor):
            embedding = embedding.numpy()

        # Reduce dimensionality using UMAP
        reducer = umap.UMAP(n_components=2)
        embedding_2d = reducer.fit_transform(embedding)

        # Known labels
        ax.scatter(embedding_2d[label_mask, 0], embedding_2d[label_mask, 1], 
                   c=labels[label_mask], cmap="Set1", s=3, alpha=0.7, label="Known Labels",
                   edgecolors='none')

        # Unknown labels
        ax.scatter(embedding_2d[~label_mask, 0], embedding_2d[~label_mask, 1], 
                   c=labels[~label_mask], cmap="Set1", s=5, alpha=0.7, 
                   label="Unknown Labels", edgecolors='black', linewidths=0.2)

        # Title with smaller font size
        pretty = pretty_map.get(embedding_type, embedding_type)
        ax.set_title(pretty, fontsize=8, pad=2)


        # Remove axis labels, ticks, and frames
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)

    # Remove empty subplots if num_embeddings is not a multiple of 4
    for j in range(i + 1, num_rows * 4):
        row, col = divmod(j, 4)
        fig.delaxes(axes[row, col])

    plt.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05, wspace=0.2, hspace=0.2)  # Adjust margins
    save_path = "./cora_analysis_results/embedding_grid_plot_cora_70_30.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Visualization saved to {save_path}")
    plt.show()

In [ ]:
def evaluate_model(true_labels, predicted_labels):
    """
    Evaluate the model's performance using accuracy, F1-score, and confusion matrix.

    Args:
        true_labels (np.array): Ground truth labels (integers).
        predicted_labels (np.array): Predicted labels (integers).

    Returns:
        dict: A dictionary containing accuracy, F1-score, and confusion matrix.
    """
    # Compute accuracy
    accuracy = accuracy_score(true_labels, predicted_labels)
    
    # Compute F1-score (macro-averaged)
    f1 = f1_score(true_labels, predicted_labels, average='macro')
    
    # Compute confusion matrix
    cm = confusion_matrix(true_labels, predicted_labels)

    #
    print(cm)
    
    # Return results as a dictionary
    return {
        'accuracy': accuracy,
        'f1_score': f1
    }

## Classifiers

In [ ]:
class NoMaskGCNConv(GCNConv):
    def compute_mask(self, inputs, mask=None):
        return None

    def call(self, inputs, training=None, mask=None):
        # Explicitly discard mask
        return super().call(inputs, mask=None)
        
class GCN(tf.keras.Model):
    def __init__(self, n_labels, seed=40):
        super().__init__()
        initializer = tf.keras.initializers.GlorotUniform(seed=seed)
        self.conv1 = NoMaskGCNConv(16, activation='relu', kernel_initializer=initializer)
        self.conv2 = NoMaskGCNConv(n_labels, activation='softmax', kernel_initializer=initializer)

    def call(self, inputs, training=False):
        x, a = inputs
        intermediate_embeddings = self.conv1([x, a])
        x = self.conv2([intermediate_embeddings, a])
        return x, intermediate_embeddings


In [ ]:
from spektral.layers import GATConv
import tensorflow as tf

# Define a custom wrapper for GATConv that avoids mask issues
class NoMaskGATConv(GATConv):
    def compute_mask(self, inputs, mask=None):
        return None

    def call(self, inputs, training=None, mask=None):
        # Explicitly discard the mask argument
        return super().call(inputs, mask=None)


# Define the GAT model using the NoMaskGATConv
class GAT(tf.keras.Model):
    def __init__(self, n_labels, num_heads=8, seed=40):
        super().__init__()
        initializer = tf.keras.initializers.GlorotUniform(seed=seed)

        # Use the custom NoMaskGATConv instead of the original GATConv
        self.conv1 = NoMaskGATConv(16, attn_heads=num_heads, concat_heads=True, activation='elu', kernel_initializer=initializer)
        self.conv2 = NoMaskGATConv(n_labels, attn_heads=1, concat_heads=False, activation='softmax', kernel_initializer=initializer)

    def call(self, inputs):
        x, a = inputs
        intermediate_embeddings = self.conv1([x, a])  # Store intermediate embeddings
        x = self.conv2([intermediate_embeddings, a])
        return x, intermediate_embeddings  # Return both final output and intermediate embeddings


In [ ]:
# Define the GraphSAGE model
class GraphSAGE(tf.keras.Model):
    def __init__(self, n_labels, hidden_dim=16, aggregator='mean', seed=40):
        super().__init__()
        initializer = tf.keras.initializers.GlorotUniform(seed=seed)

        self.conv1 = GraphSageConv(hidden_dim, activation='relu', aggregator=aggregator, kernel_initializer=initializer)
        self.conv2 = GraphSageConv(n_labels, activation='softmax', aggregator=aggregator, kernel_initializer=initializer)

    def call(self, inputs):
        x, a = inputs
        intermediate_embeddings = self.conv1([x, a])  # Store intermediate embeddings
        x = self.conv2([intermediate_embeddings, a])
        return x, intermediate_embeddings  # Return both final output and intermediate embeddings

In [ ]:
classifiers=['gcn','gat','graphsage']

## Classification using different node embeddings

In [ ]:
def train_and_evaluate(
    embedding_dict, embedding, classifier,
    ground_truth_labels=ground_truth_labels,
    masked_labels=masked_labels
):
    """
    Version that trains ONLY on the training subgraph:
    - Uses X_train and A_train
    - No test nodes seen during training
    - Inference happens on full graph
    """

    print(f"\nEmbedding: {embedding.upper()}")
    print(f"Model: {classifier.upper()}")

    # -------------------------------------------------
    # 1. Construct training subgraph
    # -------------------------------------------------
    train_mask = masked_labels != -1
    train_idx = np.where(train_mask)[0]

    # Features for training nodes only
    X_train = tf.convert_to_tensor(embedding_dict[embedding][train_mask], dtype=tf.float32)

    # Labels for training (one-hot)
    Y_train = tf.convert_to_tensor(ground_truth_labels[train_mask], dtype=tf.float32)

    # Build training adjacency
    A_train = A[train_mask][:, train_mask]   # induced subgraph

    A_coo = A_train.tocoo()
    A_train_tensor = tf.sparse.SparseTensor(
        indices = np.column_stack([A_coo.row, A_coo.col]),
        values  = A_coo.data.astype(np.float32),
        dense_shape = A_coo.shape
    )
    A_train_tensor = tf.sparse.reorder(A_train_tensor)

    # -------------------------------------------------
    # 2. Build classifier on TRAIN SUBGRAPH only
    # -------------------------------------------------
    n_classes = ground_truth_labels.shape[1]

    if classifier == 'gcn':
        model = GCN(n_classes)
    elif classifier == 'gat':
        model = GAT(n_classes)
    elif classifier == 'graphsage':
        model = GraphSAGE(n_classes)
    else:
        raise ValueError("Unknown classifier: " + classifier)

    optimizer = Adam(learning_rate=0.01)
    loss_fn = CategoricalCrossentropy()

    # -------------------------------------------------
    # 3. Training (on training subgraph only)
    # -------------------------------------------------
    print("Training on TRAINING SUBGRAPH ONLY...")

    epochs = 200
    for epoch in range(epochs):
        with tf.GradientTape() as tape:

            preds_train, _ = model([X_train, A_train_tensor], training=True)

            loss = loss_fn(Y_train, preds_train)

        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        if epoch % 20 == 0:
            acc = CategoricalAccuracy()(Y_train, preds_train).numpy()
            print(f"Epoch {epoch} | Loss={loss.numpy():.4f} | Train Acc={acc:.4f}")

    # -------------------------------------------------
    # 4. Inference — NOW use the full graph
    # -------------------------------------------------
    print("Predicting on FULL GRAPH...")

    X_full = tf.convert_to_tensor(embedding_dict[embedding], dtype=tf.float32)

    A_full = A
    A_coo = A_full.tocoo()
    A_full_tensor = tf.sparse.SparseTensor(
        indices = np.column_stack([A_coo.row, A_coo.col]),
        values  = A_coo.data.astype(np.float32),
        dense_shape = A_coo.shape
    )
    A_full_tensor = tf.sparse.reorder(A_full_tensor)

    preds_all, emb_full = model([X_full, A_full_tensor], training=False)

    predicted_labels = tf.argmax(preds_all, axis=1).numpy()

    # Evaluate only on masked nodes (test nodes)
    predicted_test = predicted_labels[labels_to_be_masked]
    true_test = labels[labels_to_be_masked]

    results = evaluate_model(true_test, predicted_test)

    print(f"Accuracy: {results['accuracy']*100:.2f}%")
    print(f"F1 Score: {results['f1_score']:.4f}")

    results["embedding"] = embedding
    results["model"] = classifier

    return results, emb_full


In [ ]:
all_results=[]
graph_embeddings_dict={}
for emb in embedding_dict.keys():
    for clf in classifiers:
        results, embedding_matrix = train_and_evaluate(embedding_dict, emb, clf)
        all_results.append(results)
        key_string= emb + ' with ' + clf
        graph_embeddings_dict[key_string]=embedding_matrix

## Saving aggregate results

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(all_results)

# Define dataset name and seed
dataset_name = "cora"
seed_value = SEED

# Save as CSV file without sorting
filename = f"{dataset_name}_70_30_{SEED}_results.csv"
filename='./cora_analysis_results/results/70_30/'+filename
df.to_csv(filename, index=False)

print(f"Results saved as {filename}")

In [ ]:
all_embeddings= embedding_dict | graph_embeddings_dict

In [ ]:
def reorder_embeddings(emb_dict, key_order):
    missing = [k for k in key_order if k not in emb_dict]
    if missing:
        print("WARNING: Missing embedding keys:", missing)

    return {k: emb_dict[k] for k in key_order if k in emb_dict}


In [ ]:
key_order = ['random', 'random with gcn', 'random with gat', 'random with graphsage', 'deepwalk', 'deepwalk with gcn', 'deepwalk with gat', 'deepwalk with graphsage', 'node2vec','node2vec with gcn', 'node2vec with gat', 'node2vec with graphsage', 'vgae', 'vgae with gcn', 'vgae with gat', 'vgae with graphsage', 'dgi', 'dgi with gcn', 'dgi with gat', 'dgi with graphsage', 'fuse', 'fuse with gcn', 'fuse with gat', 'fuse with graphsage', 'given', 'given with gcn', 'given with gat', 'given with graphsage']

In [ ]:
pretty_names = ['Random', 'Random with GCN', 'Random with GAT', 'Random with SAGE', 'Deepwalk', 'Deepwalk with GCN', 'Deepwalk with GAT', 'Deepwalk with SAGE', 'Node2Vec','Node2Vec with GCN', 'Node2Vec with GAT', 'Node2Vec with SAGE', 'VGAE', 'VGAE with GCN', 'VGAE with GAT', 'VGAE with SAGE', 'DGI', 'DGI with GCN', 'DGI with GAT', 'DGI with SAGE', 'FUSE', 'FUSE with GCN', 'FUSE with GAT', 'FUSE with SAGE', 'Given', 'Given with GCN', 'Given with GAT', 'Given with SAGE']

In [ ]:
pretty_map = dict(zip(key_order, pretty_names))

In [ ]:
print(list(all_embeddings.keys()))


In [ ]:
all_embeddings = reorder_embeddings(all_embeddings, key_order)

In [ ]:
visualize_all_embeddings(all_embeddings, labels, label_mask)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    adjusted_rand_score,
    v_measure_score
)
import pandas as pd
# from benchmarking_utils import load_dataset  # or your label loader

# Load true labels
# ds = load_dataset("cora")
# labels = ds["labels"]
ds=dataset
# Choose which embedding sources to evaluate
sources = {
    "Raw Embeddings": embedding_dict,
    "GNN Embeddings": graph_embeddings_dict  # << the key fix
}

metrics_results = []

for source_name, emb_source in sources.items():
    print(f"\n Evaluating {source_name}...\n")

    for name, emb_tensor in emb_source.items():
        emb = emb_tensor.numpy() if isinstance(emb_tensor, tf.Tensor) else np.array(emb_tensor)

        n_clusters = len(np.unique(labels))
        kmeans = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10)
        cluster_labels = kmeans.fit_predict(emb)

        try:
            silhouette = silhouette_score(emb, cluster_labels)
        except Exception:
            silhouette = np.nan
        try:
            db_index = davies_bouldin_score(emb, cluster_labels)
        except Exception:
            db_index = np.nan

        nmi = normalized_mutual_info_score(labels, cluster_labels)
        ari = adjusted_rand_score(labels, cluster_labels)
        v_measure = v_measure_score(labels, cluster_labels)

        metrics_results.append({
            "Source": source_name,
            "Embedding": name,
            "Silhouette": silhouette,
            "Davies-Bouldin": db_index,
            "NMI": nmi,
            "ARI": ari,
            "V-Measure": v_measure
        })

        print(f" {name.upper()} | Silhouette={silhouette:.4f}, DB={db_index:.4f}, NMI={nmi:.4f}, ARI={ari:.4f}, V={v_measure:.4f}")

# Save results
metrics_df = pd.DataFrame(metrics_results)
metrics_path = os.path.join(save_dir, f"clustering_metrics_combined_{dataset_name}_{split_folder.replace('-', '_')}_{SEED}.csv")
metrics_df.to_csv(metrics_path, index=False)
print(f"\n Combined clustering metrics saved to: {metrics_path}")
print(metrics_df)


In [ ]:
import pandas as pd
import os
import numpy as np

# ----------------------------------------
# Configuration
# ----------------------------------------
dataset_name = "cora"
split_folder = "30_70"
seeds = [42, 46, 123, 999, 2025]
base_dir = f"./cora_analysis_results/embeddings/{split_folder}/"

# Output file
output_path = os.path.join(base_dir, f"clustering_metrics_combined_{dataset_name}_{split_folder}_averaged.csv")

# ----------------------------------------
# Load and combine all CSVs
# ----------------------------------------
dfs = []
for seed in seeds:
    filename = f"clustering_metrics_combined_{dataset_name}_{split_folder}_{seed}.csv"
    filepath = os.path.join(base_dir, filename)
    
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        df["Seed"] = seed
        dfs.append(df)
        print(f" Loaded: {filepath}")
    else:
        print(f" Missing file for seed {seed}: {filepath}")

if not dfs:
    raise FileNotFoundError("No CSV files found — check your directory and filenames.")

# Merge all seed data
combined_df = pd.concat(dfs, ignore_index=True)

# ----------------------------------------
# Average metrics across seeds
# ----------------------------------------
agg_df = (
    combined_df
    .groupby(["Source", "Embedding"])
    .agg({
        "Silhouette": ["mean", "std"],
        "Davies-Bouldin": ["mean", "std"],
        "NMI": ["mean", "std"],
        "ARI": ["mean", "std"],
        "V-Measure": ["mean", "std"]
    })
    .reset_index()
)

# Flatten column names
agg_df.columns = [
    "Source", "Embedding",
    "Silhouette_Mean", "Silhouette_STD",
    "DB_Mean", "DB_STD",
    "NMI_Mean", "NMI_STD",
    "ARI_Mean", "ARI_STD",
    "VMeasure_Mean", "VMeasure_STD"
]

# Save the averaged results
agg_df.to_csv(output_path, index=False)
print(f"\n Averaged metrics saved to: {output_path}\n")

# Display summary
print(agg_df)
